## Robustness of networks using MILP and Fast-Lin

### Analyzes how the robustness of networks depends on variables such as dense vs CNN, number of layers, and width of layers.

Let $f: [0, 1]^{n_0} \to \mathbb{R}^{10}$ denote a neural network trained to classify numbers 0-9, where the network classifies image $x$ being labelled as $i$ more likely than $j$ if $f_i(x) > f_j(x)$.

Let the image $x_0$ be classified as $i$ by the network. We find non-trivial $\epsilon$ such that $f_i(x) > f_j(x)$ for all $x \in B_p(x_0, \epsilon)$ and $i \neq j$.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [2]:
import csv
from matplotlib import pyplot as plt
import pandas as pd
from pathlib import Path
import torch
from torchvision import datasets
from tqdm import tqdm

from src.architectures.networkArchitectures import networkRegistry
from src.robustness.exactRobustness import exactRobustness
from src.robustness.fastLin import fastLin
from src.testing.networkTesting import testNetwork
from src.training.networkTraining import transform
from src.utils.extractParams import extractParams
from src.utils.loadNetwork import loadNetwork

In [3]:
testingSet = datasets.MNIST(root = "./data", train = False, transform = transform, download = True)
numberOfImages = 1    # Number of images to analyze for each network
pNorm = 1               # Norm used for robustness

### Computes certified lower bounds

Gets certified lower bounds by Fast-Lin. Could be fast enough for all considered networks even unoptimized. Computed results are saved in the results folder. This allows us to use these without computing them every time.

In [4]:
# Results are saved in a csv file. Columns are give by:
# "name", the name of the network
# "imageIndex", the index of the image
# "label", the true label of the image
# "predictedClass", the predicted label of the image
# "pNorm", the norm used for robustness
# "exactRobustness", whether robustness was computed by MILP or Fast-Lin
# "epsilon", the exact epsilon for exactRobustness and the certified lower bound otherwise

resultsDirectory = Path("results2")
fileName = "robustnessResults.csv"
filePath = resultsDirectory / fileName

# Load already completed results
if filePath.exists():
    df = pd.read_csv(filePath)
    computedResults = set(zip(df["name"], df["imageIndex"], df["pNorm"], df["exactRobustness"]))
else:
    computedResults = set()

# Open CSV for appending
fileExists = filePath.exists()

with filePath.open(mode = "a", newline = "") as file:
    writer = csv.writer(file)

    # Create header if file does not already exist
    if not fileExists:
        writer.writerow(["name", "imageIndex", "label", "predictedClass", "pNorm", "exactRobustness", "epsilon"])

    for networkName in networkRegistry:
        print(f"Computing Fast-Lin for network: {networkName} (pNorm = {pNorm})")
        network = loadNetwork(networkName)

        parameters = extractParams(network = network, inputShape = (1, 28, 28))
        weights = [W for W, _ in parameters]
        biases = [b for _, b in parameters]

        for imageIndex in tqdm(range(numberOfImages)):
            # Skip image if already computed for the network using Fast-Lin
            if (networkName, imageIndex, pNorm, False) in computedResults:
                continue

            # Compute results
            image, label = testingSet[imageIndex]
            x0 = image.view(-1).numpy()

            output = network(image)
            _, predictedClass = torch.max(output, 1)
            predictedClass = predictedClass.item()
            targetClasses = [targetClass for targetClass in range(0, 10) if targetClass != predictedClass]
            
            certifiedEpsilon, _, _ = fastLin(weights = weights, biases = biases, x0 = x0, pNorm = pNorm, epsilon0 = 10, originalClass = predictedClass, targetClasses = targetClasses, tolerance = 0.005)

            # Save computed results
            writer.writerow([networkName, imageIndex, label, predictedClass, pNorm, False, certifiedEpsilon])
            file.flush()

Computing Fast-Lin for network: Dense1x60 (pNorm = 1)


100%|██████████| 1/1 [00:00<00:00,  2.74it/s]


Computing Fast-Lin for network: Dense2x30 (pNorm = 1)


100%|██████████| 1/1 [00:00<00:00,  6.65it/s]


Computing Fast-Lin for network: Dense3x20 (pNorm = 1)


...

100%|██████████| 1/1 [00:01<00:00,  1.09s/it]


Computing Fast-Lin for network: Dense3x1000 (pNorm = 1)


100%|██████████| 1/1 [00:11<00:00, 11.62s/it]


Computing Fast-Lin for network: LeNet (pNorm = 1)


100%|██████████| 1/1 [01:12<00:00, 72.86s/it]


Plot figures comparing robustness vs layer width, number of layers, and total number of neurons.

In [ ]:
# Plot robustness obtained by Fast-Lin

# Load data
df = pd.read_csv(filePath)

# Filter results obtained by Fast-Lin
dfFastLin = df[df["exactRobustness"] == False]
dfFastLin_p = dfFastLin[dfFastLin["pNorm"] == pNorm]

networkNames = dfFastLin_p["name"].unique()

# Plot robustness vs depth
depths = []
averageCertifiedEpsilons = []
for networkName in networkNames:
    networkEntry = networkRegistry[networkName]

    if "20width" not in networkEntry.tags:
        continue

    if networkName == "Dense2x20": depth = 2
    elif networkName == "Dense3x20": depth = 3
    elif networkName == "Dense4x20": depth = 4
    elif networkName == "Dense5x20": depth = 5
    else:
        print("Warning, unexpected happened!")

    dfNetwork = dfFastLin_p[dfFastLin_p["name"] == networkName]
    averageCertifiedEpsilon = dfNetwork["epsilon"].mean()

    depths.append(depth)
    averageCertifiedEpsilons.append(averageCertifiedEpsilon)

plt.figure()
x, y = zip(*sorted(zip(depths, averageCertifiedEpsilons)))  # Sort before plotting
plt.title("Robustness vs depth")
plt.plot(x, y, "-o")
plt.xlabel("Network depth")
plt.ylabel("Average certified lower bound")
plt.savefig("results/robustnessVsDepth.eps", format = "eps")
plt.show()

# Plot robustness vs width
widths = []
averageCertifiedEpsilons = []
for networkName in networkNames:
    networkEntry = networkRegistry[networkName]

    if "3layers" not in networkEntry.tags:
        continue

    if networkName == "Dense3x20": width = 20
    elif networkName == "Dense3x50": width = 50
    elif networkName == "Dense3x200": width = 200
    elif networkName == "Dense3x1000": width = 1000
    else:
        print("Warning, unexpected happened!")

    dfNetwork = dfFastLin_p[dfFastLin_p["name"] == networkName]
    averageCertifiedEpsilon = dfNetwork["epsilon"].mean()

    widths.append(width)
    averageCertifiedEpsilons.append(averageCertifiedEpsilon)

plt.figure()
x, y = zip(*sorted(zip(widths, averageCertifiedEpsilons)))  # Sort before plotting
plt.plot(x, y, "-o")
plt.title("Robustness vs width")
plt.xlabel("Network width")
plt.ylabel("Average certified lower bound")
plt.savefig("results/robustnessVsWidth.eps", format = "eps")
plt.show()

# Plot robustness vs accuracy
accuracies = []
averageCertifiedEpsilons = []
for networkName in networkNames:
    networkEntry = networkRegistry[networkName]

    if networkName == "LeNet":
        continue

    network = loadNetwork(networkName)
    correctClassifications, totalClassifications = testNetwork(network = network)
    sumOfCorrectClassifications = sum(correctClassifications.values())
    sumOfTotalClassifications = sum(totalClassifications.values())
    accuracy = 100 * float(sumOfCorrectClassifications) / sumOfTotalClassifications

    dfNetwork = dfFastLin_p[dfFastLin_p["name"] == networkName]
    averageCertifiedEpsilon = dfNetwork["epsilon"].mean()

    accuracies.append(accuracy)
    averageCertifiedEpsilons.append(averageCertifiedEpsilon)

plt.figure()
x, y = zip(*sorted(zip(accuracies, averageCertifiedEpsilons)))  # Sort before plotting
plt.plot(x, y, "-o")
plt.title("Robustness vs accuracy")
plt.xlabel("Network accuracy")
plt.ylabel("Average certified lower bound")
plt.savefig("results/robustnessVsAccuracy.eps", format = "eps")
plt.show()

### Computes exact robustness

Get exact robustness by solving multiple MILPs created via the big-M formulation. Infeasible for larger networks.

In [ ]:
# Results are saved in a csv file. Columns are give by:
# "name", the name of the network
# "imageIndex", the index of the image
# "label", the true label of the image
# "predictedClass", the predicted label of the image
# "pNorm", the norm used for robustness
# "exactRobustness", whether robustness was computed by MILP or Fast-Lin
# "epsilon", the exact epsilon for exactRobustness and the certified lower bound otherwise

resultsDirectory = Path("results")
fileName = "robustnessResults.csv"
filePath = resultsDirectory / fileName

# Load already completed results
if filePath.exists():
    df = pd.read_csv(filePath)
    computedResults = set(zip(df["name"], df["imageIndex"], df["pNorm"], df["exactRobustness"]))
else:
    computedResults = set()

# Open CSV for appending
fileExists = filePath.exists()

with filePath.open(mode = "a", newline = "") as file:
    writer = csv.writer(file)

    # Create header if file does not already exist
    if not fileExists:
        writer.writerow(["name", "imageIndex", "label", "predictedClass", "pNorm", "exactRobustness", "epsilon"])

    for networkName in networkRegistry:
        if "60neurons" not in networkRegistry[networkName].tags:
            continue

        print(f"Computing MILP for network: {networkName} (pNorm = {pNorm})")
        network = loadNetwork(networkName)

        parameters = extractParams(network = network, inputShape = (1, 28, 28))
        weights = [W for W, _ in parameters]
        biases = [b for _, b in parameters]

        for imageIndex in tqdm(range(numberOfImages)):
            # Skip image if already computed for the network using MILP
            if (networkName, imageIndex, pNorm, True) in computedResults:
                continue

            # Compute results
            image, label = testingSet[imageIndex]
            x0 = image.view(-1).numpy()

            output = network(image)
            _, predictedClass = torch.max(output, 1)
            predictedClass = predictedClass.item()
            targetClasses = [targetClass for targetClass in range(0, 10) if targetClass != predictedClass]
            
            epsilon, _, _ = exactRobustness(weights = weights, biases = biases, x0 = x0, pNorm = pNorm, epsilon0 = 10, originalClass = predictedClass, targetClasses = targetClasses)

            # Save computed results
            writer.writerow([networkName, imageIndex, label, predictedClass, pNorm, True, epsilon])
            file.flush()

Plot figures comparing robustness vs layer width, number of layers, and total number of neurons for some smaller networks.

In [ ]:
# Plot robustness obtained by MILP

# Load data
df = pd.read_csv(filePath)

# Filter results obtained by MILP
dfMILP = df[df["exactRobustness"] == True]
dfMILP_p = dfMILP[dfMILP["pNorm"] == pNorm]

networkNames = dfMILP_p["name"].unique()
for networkName in networkNames:
    subset = dfMILP_p[dfMILP_p["name"] == networkName]

    plt.figure()
    plt.hist(subset["epsilon"], bins = 15)
    plt.title(f"Certified lower bounds histogram for {networkName} (p = {pNorm})")
    plt.xlabel("Certified lower bound")
    plt.ylabel("Count")
    plt.show()